In [2]:
# Setting up RAG

# Set up the OpenAI client:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Load the data and build the search index:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

# Create the assistant:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [3]:
# Testing it

# Let's try a question:
assistant.rag("How do I run Ollama locally?")

# This works fine. The search finds relevant FAQ entries about Ollama, and the LLM gives a good answer.

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download  \n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo test that the local server is running, you can also run:\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [4]:
# Now try something slightly different:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about running **Ollama** locally.\n\nThe closest related note is that you **can run the course locally** instead of Codespaces if you’re comfortable setting up the needed tools like Python, `uv`, Jupyter, Docker, and others for the module. If you do run locally, document your setup and keep it reproducible.\n\nIf you meant something else by “Olama,” let me know.'

In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I still join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Yes — probably, but it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure out the best next step. Usually you should:\n\n1. Check the course page for the enrollment deadline or “late registration” info.\n2. Contact the instructor or course administrator ASAP.\n3. Ask whether there’s a waitlist or an option to join late.\n\nIf you want, I can also help you draft a short message asking to join the course.'

In [7]:
# now instead of sending only a question to the lllm well also give him the function he can use

# defining the tool: 

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search('how do i run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [9]:
# llms are language agnostic. we just making an http call to some endpoint, and we need 
# a language agnostic way of defining a function:

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)


In [12]:
len(response.output)

1

In [13]:
call = response.output[0]

In [20]:
# llm already changed the question (this is not the question in the way we)
call

ResponseFunctionToolCall(arguments='{"query":"Can I still join the course if I just discovered it?"}', call_id='call_gXm21zJlQrnuyOheUMl8IyN3', name='search', type='function_call', id='fc_0dd44feb79ae0ed6006a3952cbf6fc819198843a4be48ce10f', namespace=None, status='completed')

In [ ]:
# this was the tool he is using
call.name

'search'

In [ ]:
# now we need to parse these arguments

import json

# this is the query he is using 
args = json.loads(call.arguments)

In [25]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

In [ ]:
# what happenend?
# history


# 1. make a call to te LLM <-- we pay
# 2. LLM decided to invoke search('params')
# 3. we invoke the search , we have the results 
# 4. send the results back to the llm (another call) <-- we pay again
# 5. llm processes the results 
# 6. llm gives the answer

In [30]:
# -> when we turn rag into agentic rag, we need to do more!

# turn everything into json what well send to the llm

result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "9f689c185f",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I missed the first homework - can I still get a certificate?",
    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",
    "answer": "You don't 

In [31]:
# we received a functionToolCall response, a decision of an llm to use a tool
# basically: he told us to invoke the function, we used the function,  
# there i a related call id (bc we need to know for which call it was needed)

function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [ ]:
# llms are stateless! every time they need the entire previous history!

# append decision
messages.append(call)

# append output
messages.append(function_call_output)

print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I still join it?'}, ResponseFunctionToolCall(arguments='{"query":"Can I still join the course if I just discovered it?"}', call_id='call_gXm21zJlQrnuyOheUMl8IyN3', name='search', type='function_call', id='fc_0dd44feb79ae0ed6006a3952cbf6fc819198843a4be48ce10f', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_gXm21zJlQrnuyOheUMl8IyN3', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n    "answer": "Yes, you

In [33]:
# send results back to the llm to process the results 

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [34]:
# what could happen now is: either it actually has an answer for us, or it decides to make another searhch because it doesnt have enough information


print(response.output_text)

Yes — you can still join and start learning anytime.

If you want a certificate, the key thing is to submit your project while submissions are still open.


In [ ]:
# summary: the path was a bit longer but now the agent is in control and can decide whats happening with the user 
# query and what to do with the too: now the llm is thedriver!

# that is what makes the call to the llm agentic!

# agent has a certain task: (in this case to help the user student)
# agent has memory
# agent has tools (to actually carry out the task)



In [38]:
# how much we payed for this? 

usage = response.usage
print(usage.input_tokens, usage.output_tokens) 

prompt = """ 
i use gpt 5-4 min
(814,35)
this is input output tokens
write a python function to calculate the price and also give the price
"""

814 35


In [39]:
def calculate_gpt54_mini_cost(input_tokens: int, output_tokens: int):
    INPUT_PRICE_PER_MILLION = 0.25   # USD
    OUTPUT_PRICE_PER_MILLION = 2.00  # USD

    input_cost = input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
    output_cost = output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


cost = calculate_gpt54_mini_cost(814, 35)
print(cost)

{'input_cost': 0.0002035, 'output_cost': 7e-05, 'total_cost': 0.0002735}


In [ ]:
# THE AGENTIC LOOP

# What if the LLM decides to make multiple calls?

# what happenend?
# history


# 1. make a call to te LLM <-- we pay
# 2. LLM decided to invoke search('params')
# 3. we invoke the search , we have the results 
# 4. send the results back to the llm (another call) <-- we pay again
# 5. llm processes the results 
# 6. LLM decides to make another tool call
# 7. send one more api request 
# 8. process and gives the answer 
# ...

# for that, we need: 
# 1. developer prompt: the role and behavior
# ...

# -> agent(instructions, memory, tools)

# we exit the loop when there are no more tool calls 


In [ ]:
# setup

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

